# LAB3 - Modelowanie danych w hurtowni

Notebook realizuje laboratorium 3 dla zbioru `Online_Retail.csv`. Celem jest przygotowanie schematu gwiazdy, określenie ziarna danych, zastosowanie kluczy sztucznych oraz implementacja SCD typu 2 dla wymiaru klienta.

Rozwiązanie obejmuje również część dodatkową: dodanie dodatkowych wymiarów, rozszerzenie tabeli faktów o miary oraz krótką analizę jakości projektu.

## 0. Import bibliotek i konfiguracja ścieżek

Używamy biblioteki `pandas` do przetwarzania danych oraz `matplotlib` do przygotowania wykresu trendu sprzedaży. Plik wejściowy powinien znajdować się w katalogu `data/Online_Retail.csv`, a wyniki zostaną zapisane do katalogu `output/`.

In [ ]:
import matplotlib
matplotlib.use("Agg")

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_DIR / "Online_Retail.csv"

## 1. Wczytanie danych - staging

Wczytujemy dane źródłowe do DataFrame. Plik `Online_Retail.csv` wymaga kodowania `ISO-8859-1`, ponieważ zawiera znaki, które mogą nie zostać poprawnie odczytane przy domyślnym kodowaniu UTF-8. Na tym etapie dane traktujemy jako staging, czyli surową warstwę wejściową hurtowni.

In [ ]:
df_raw = pd.read_csv(INPUT_FILE, encoding="ISO-8859-1")

print("Liczba rekordów i kolumn:", df_raw.shape)
display(df_raw.head())
df_raw.info()

## 2. Czyszczenie danych

Zgodnie z wymaganiami usuwamy brakujące `CustomerID`, anulowane faktury, rekordy z `Quantity <= 0`, rekordy z `UnitPrice <= 0`, konwertujemy datę oraz usuwamy duplikaty. Następnie tworzymy miarę `Revenue = Quantity * UnitPrice`.

Takie czyszczenie jest konieczne, ponieważ błędne rekordy mogłyby zafałszować raporty sprzedażowe.

In [ ]:
df = df_raw.copy()

before_rows = len(df)

df = df.dropna(subset=["CustomerID"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df = df.dropna(subset=["InvoiceDate"])

df = df.drop_duplicates()

df["CustomerID"] = df["CustomerID"].astype(int)
df["StockCode"] = df["StockCode"].astype(str)
df["Description"] = df["Description"].fillna("Unknown product")
df["Country"] = df["Country"].fillna("Unknown country")

df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceDateOnly"] = df["InvoiceDate"].dt.date

print("Liczba rekordów przed czyszczeniem:", before_rows)
print("Liczba rekordów po czyszczeniu:", len(df))
display(df.head())

## 3. Wybór grain

Wybrano grain na poziomie pojedynczej pozycji faktury. Jeden rekord w tabeli faktów `FactSales` odpowiada jednej linii sprzedażowej z faktury, czyli sprzedaży konkretnego produktu w konkretnej fakturze.

Ten wybór daje największą elastyczność analityczną: można analizować sprzedaż według produktu, klienta, kraju i czasu, a następnie agregować dane do poziomu faktury, dnia, miesiąca lub kraju. Przykład analizy: sprzedaż produktów według kraju i miesiąca.

## 4. Klucze naturalne i sztuczne

Klucze naturalne pochodzą z danych źródłowych, np. `CustomerID`, `StockCode`, `InvoiceNo`, `Country` oraz data z `InvoiceDate`. W hurtowni danych wprowadzamy jednak klucze sztuczne, np. `customer_key`, `product_key`, `date_key`, `country_key`, `invoice_key`.

Tabela faktów będzie używać kluczy sztucznych, ponieważ są stabilniejsze i wygodniejsze w modelu gwiazdy, szczególnie przy wersjonowaniu SCD.

## 5. Budowa wymiarów

Tworzymy wymiary: `DimCountry`, `DimProduct`, `DimDate`, `DimInvoice` oraz `DimCustomer`. Minimalne wymagania obejmują `DimCustomer`, `DimProduct` i `DimDate`, natomiast w części dodatkowej rozszerzamy model o `DimCountry` oraz `DimInvoice`.

In [ ]:
# DimCountry
DimCountry = (
    df[["Country"]]
    .drop_duplicates()
    .sort_values("Country")
    .reset_index(drop=True)
)
DimCountry.insert(0, "country_key", range(1, len(DimCountry) + 1))
DimCountry = DimCountry.rename(columns={"Country": "country_name"})

# DimProduct
DimProduct = (
    df.groupby("StockCode", as_index=False)
    .agg(product_name=("Description", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown product"))
    .sort_values("StockCode")
    .reset_index(drop=True)
)
DimProduct.insert(0, "product_key", range(1, len(DimProduct) + 1))
DimProduct = DimProduct.rename(columns={"StockCode": "stock_code"})

# DimDate
DimDate = (
    df[["InvoiceDateOnly"]]
    .drop_duplicates()
    .sort_values("InvoiceDateOnly")
    .reset_index(drop=True)
)
DimDate.insert(0, "date_key", range(1, len(DimDate) + 1))
DimDate = DimDate.rename(columns={"InvoiceDateOnly": "date"})
DimDate["date"] = pd.to_datetime(DimDate["date"])
DimDate["year"] = DimDate["date"].dt.year
DimDate["quarter"] = DimDate["date"].dt.quarter
DimDate["month"] = DimDate["date"].dt.month
DimDate["day"] = DimDate["date"].dt.day
DimDate["weekday"] = DimDate["date"].dt.day_name()

# DimInvoice
DimInvoice = (
    df[["InvoiceNo", "InvoiceDate"]]
    .drop_duplicates(subset=["InvoiceNo"])
    .sort_values("InvoiceNo")
    .reset_index(drop=True)
)
DimInvoice.insert(0, "invoice_key", range(1, len(DimInvoice) + 1))
DimInvoice = DimInvoice.rename(columns={"InvoiceNo": "invoice_no", "InvoiceDate": "invoice_datetime"})
DimInvoice["invoice_date"] = DimInvoice["invoice_datetime"].dt.date
DimInvoice["invoice_hour"] = DimInvoice["invoice_datetime"].dt.hour

print("DimCountry:", DimCountry.shape)
print("DimProduct:", DimProduct.shape)
print("DimDate:", DimDate.shape)
print("DimInvoice:", DimInvoice.shape)

display(DimCountry.head())
display(DimProduct.head())
display(DimDate.head())
display(DimInvoice.head())

## 6. SCD typu 2 dla DimCustomer

Dla wymiaru klienta implementujemy SCD typu 2. Przyjęto, że zmianą śledzoną historycznie jest zmiana kraju klienta. Jeżeli ten sam `CustomerID` pojawi się z innym krajem w późniejszym czasie, powstaje nowa wersja rekordu.

Wymiar zawiera kolumny `valid_from`, `valid_to` oraz `current_flag`, co pozwala zachować historię zmian i jednocześnie wskazać aktualny rekord.

In [ ]:
customer_country_history = (
    df[["CustomerID", "Country", "InvoiceDate"]]
    .drop_duplicates()
    .sort_values(["CustomerID", "InvoiceDate"])
    .reset_index(drop=True)
)

scd_records = []

for customer_id, customer_rows in customer_country_history.groupby("CustomerID"):
    customer_rows = customer_rows.sort_values("InvoiceDate")
    current_country = None
    valid_from = None

    for _, row in customer_rows.iterrows():
        row_country = row["Country"]
        row_date = row["InvoiceDate"]

        if current_country is None:
            current_country = row_country
            valid_from = row_date
        elif row_country != current_country:
            scd_records.append(
                {
                    "customer_id": int(customer_id),
                    "country_name": current_country,
                    "valid_from": valid_from,
                    "valid_to": row_date,
                    "current_flag": False,
                }
            )
            current_country = row_country
            valid_from = row_date

    scd_records.append(
        {
            "customer_id": int(customer_id),
            "country_name": current_country,
            "valid_from": valid_from,
            "valid_to": pd.NaT,
            "current_flag": True,
        }
    )

DimCustomer = pd.DataFrame(scd_records)
DimCustomer = DimCustomer.merge(DimCountry, on="country_name", how="left")
DimCustomer.insert(0, "customer_key", range(1, len(DimCustomer) + 1))
DimCustomer = DimCustomer[
    [
        "customer_key",
        "customer_id",
        "country_key",
        "country_name",
        "valid_from",
        "valid_to",
        "current_flag",
    ]
]

print("DimCustomer SCD2:", DimCustomer.shape)
display(DimCustomer.head())

## 7. Budowa tabeli faktów FactSales

Tabela faktów zawiera miary: `quantity`, `unit_price`, `revenue` i `line_count`. Używa wyłącznie kluczy sztucznych do wymiarów: `customer_key`, `product_key`, `date_key`, `country_key`, `invoice_key`.

Dzięki temu model ma postać gwiazdy i jest wygodny do analiz OLAP.

In [ ]:
# Dodanie customer_key z uwzględnieniem zakresu ważności SCD2
fact_base = df.merge(
    DimCustomer[["customer_key", "customer_id", "country_name", "valid_from", "valid_to"]],
    left_on=["CustomerID", "Country"],
    right_on=["customer_id", "country_name"],
    how="left",
)

in_valid_period = (
    (fact_base["InvoiceDate"] >= fact_base["valid_from"])
    & (
        fact_base["valid_to"].isna()
        | (fact_base["InvoiceDate"] < fact_base["valid_to"])
    )
)

fact_base = fact_base[in_valid_period].copy()

fact_base = fact_base.merge(
    DimProduct[["product_key", "stock_code"]],
    left_on="StockCode",
    right_on="stock_code",
    how="left",
)

fact_base = fact_base.merge(
    DimCountry[["country_key", "country_name"]],
    left_on="Country",
    right_on="country_name",
    how="left",
)

DimDate_for_merge = DimDate.copy()
DimDate_for_merge["date_only"] = DimDate_for_merge["date"].dt.date

fact_base = fact_base.merge(
    DimDate_for_merge[["date_key", "date_only"]],
    left_on="InvoiceDateOnly",
    right_on="date_only",
    how="left",
)

fact_base = fact_base.merge(
    DimInvoice[["invoice_key", "invoice_no"]],
    left_on="InvoiceNo",
    right_on="invoice_no",
    how="left",
)

FactSales = fact_base[
    [
        "invoice_key",
        "customer_key",
        "product_key",
        "date_key",
        "country_key",
        "Quantity",
        "UnitPrice",
        "Revenue",
    ]
].copy()

FactSales.insert(0, "sales_key", range(1, len(FactSales) + 1))
FactSales["line_count"] = 1
FactSales = FactSales.rename(
    columns={
        "Quantity": "quantity",
        "UnitPrice": "unit_price",
        "Revenue": "revenue",
    }
)

print("FactSales:", FactSales.shape)
display(FactSales.head())

## 8. Analizy biznesowe

Model pozwala odpowiadać na pytania biznesowe, np. jaka jest sprzedaż według kraju oraz jak zmieniała się sprzedaż w czasie. Wykonujemy dwie przykładowe agregacje i zapisujemy wyniki do plików CSV.

In [ ]:
sales_by_country = (
    FactSales.merge(DimCountry, on="country_key", how="left")
    .groupby("country_name")
    .agg(
        revenue=("revenue", "sum"),
        quantity=("quantity", "sum"),
        lines=("line_count", "sum"),
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

sales_trend = (
    FactSales.merge(DimDate, on="date_key", how="left")
    .groupby(["year", "month"])
    .agg(
        revenue=("revenue", "sum"),
        quantity=("quantity", "sum"),
        lines=("line_count", "sum"),
    )
    .reset_index()
    .sort_values(["year", "month"])
)

sales_by_product = (
    FactSales.merge(DimProduct, on="product_key", how="left")
    .groupby(["stock_code", "product_name"])
    .agg(revenue=("revenue", "sum"), quantity=("quantity", "sum"))
    .reset_index()
    .sort_values("revenue", ascending=False)
)

display(sales_by_country.head(10))
display(sales_trend.head())
display(sales_by_product.head(10))

## 9. Zapis wyników

Zapisujemy tabele wymiarów, tabelę faktów, pliki analityczne oraz wykres trendu. Te pliki stanowią wynik działania modelu hurtowni danych.

In [ ]:
DimCustomer.to_csv(OUTPUT_DIR / "DimCustomer_SCD2.csv", index=False)
DimProduct.to_csv(OUTPUT_DIR / "DimProduct.csv", index=False)
DimDate.to_csv(OUTPUT_DIR / "DimDate.csv", index=False)
DimCountry.to_csv(OUTPUT_DIR / "DimCountry.csv", index=False)
DimInvoice.to_csv(OUTPUT_DIR / "DimInvoice.csv", index=False)
FactSales.to_csv(OUTPUT_DIR / "FactSales.csv", index=False)

sales_by_country.to_csv(OUTPUT_DIR / "sales_by_country.csv", index=False)
sales_trend.to_csv(OUTPUT_DIR / "sales_trend_monthly.csv", index=False)
sales_by_product.to_csv(OUTPUT_DIR / "sales_by_product.csv", index=False)

sales_trend["period"] = (
    sales_trend["year"].astype(str)
    + "-"
    + sales_trend["month"].astype(str).str.zfill(2)
)

plt.figure(figsize=(12, 6))
plt.plot(sales_trend["period"], sales_trend["revenue"], marker="o")
plt.title("Trend sprzedaży w czasie")
plt.xlabel("Miesiąc")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sales_trend_monthly.png")
plt.show()

print("Zapisano wyniki do:", OUTPUT_DIR)

## 10. Analiza projektu i wnioski

Wybrano ziarno na poziomie pozycji faktury, ponieważ daje największą szczegółowość i umożliwia późniejsze agregowanie danych do poziomu faktury, dnia, miesiąca, klienta, produktu lub kraju. Wadą tego rozwiązania jest większy rozmiar tabeli faktów, ale zyskujemy elastyczność analityczną.

W modelu zastosowano kompromis charakterystyczny dla Kimbala: dane są częściowo zdenormalizowane, aby uprościć analizy OLAP. W porównaniu z modelem 3NF zapytania są prostsze, bo tabela faktów łączy się bezpośrednio z wymiarami.

Model wspiera analizy biznesowe, ponieważ pozwala badać przychód, ilość sprzedanych produktów i liczbę linii sprzedaży według czasu, kraju, klienta, produktu oraz faktury. SCD typu 2 w `DimCustomer` pozwala zachować historię zmian kraju klienta.